# TIM 175 RAG Evaluation using Rag Metrics

Use this Collab for the Task 3 to evaluate user responses with RAGAS metrics

In [ ]:
#You do not need to edit this cell. Just run it using the button on the left
%%capture
!pip install ragas
!pip install langchain_google_genai

import pandas as pd
import os
import ast
from google.colab import userdata, files

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

## Step 1. Loading in our Documents

We will first load our data in through a CSV file of our RAG results and Evaluation Google Sheets. You can download a CSV file of your results:

- 1) Open your Results and Evaluation Spreadsheet
- 2) Click on File and Download
- 3) Choose Comma Separated Values (.csv)

In [ ]:
# Get the filename
csv_rag_results = files.upload()
filename = next(iter(csv_rag_results))

# Read the raw file contents
with open(filename, 'r') as f:
    lines = f.readlines()

# Find the header row with "User query"
for i, line in enumerate(lines):
    if "User query" in line:
        header_index = i
        break

# Load CSV starting from the detected header row
df = pd.read_csv(filename, skiprows=header_index)

# Drop rows where all values are NaN
df = df.dropna(subset=[
    "User query",
    "Retrieved context string from the Google Colab",
    "User response produced by Prompt 2"
])
df

Saving Group 16 8. RAG Results & Evaluation - RAGAS Evaluation.csv to Group 16 8. RAG Results & Evaluation - RAGAS Evaluation.csv


,User query,Which podcast episode you know contains relevant content (if applicable),The relevant content you know exists in that podcast episode (if applicable),Query object produced by Prompt 1,Retrieved context string from the Google Colab,User response produced by Prompt 2,Response Relevancy,Faithfulness,Reflection on RAGAS Metrics for User Response
0,Example Query: What skills and training did yo...,078_Kayla Baumgardner Firefighter Paramedic.txt,"Interviewer 1:02 \nYeah, our pleasure. Kayla...","{\n ""content_string_query"": ""What skills and ...",Top 4 Most Relevant Excerpts:'\n{'Passage': 'o...,"To become a firefighter, one must undergo sign...",0.8351,0.8049,NaN
21,What kind of personality do you need to work i...,144_Chloe Woodmansee _ City Clerk City of Capi...,[00:19:47] Interviewer: \nvery cool people. Ch...,"{\n ""content_string_query"": ""What personality...","[{'Passage': ""Dante Searcy 23:02 Yes. So I wo...","Okay, I understand you're curious about the ki...",NaN,NaN,NaN
27,I'm passionate about giving back and would lov...,154_PatriciaGreenway_SantaCruzCitySchoolsCaree...,Patrick Hart 8:06 \r\nNice. Do you want to t...,"{\n ""content_string_query"": ""Ways to contribu...","[{'Passage': ""And so we provide A variety of e...","Okay, I understand you're looking for meaningf...",0.8661,1.0000,High relevancy score indicates that retrived p...
29,I'm feeling really stressed about my future an...,166_DanteSearcy_HRGeneralist.txt,"Interviewer 18:33 \r\nYeah, lots of learning...","{\n ""content_string_query"": ""Strategies for m...","[{'Passage': ""Melina 26:39 The last question w...",I understand how stressful it can be to worry ...,0.8808,1.0000,High faithfulness score indicates that the res...


# Step 2. Collect Evaluation Data

To collect evaluation data, we first need a set of queries to run against our RAG. You already have these from the RAG results and Evaluation Google Sheets.
We will load this into an EvaluationDataset object from Ragas to be able to start our evaluation.

In [ ]:
#You do not need to edit this cell. Just run it using the button on the left
queries = df["User query"].tolist()
retrieved_contexts = df["Retrieved context string from the Google Colab"].tolist()
retrieved_contexts = [[i] for i in retrieved_contexts]
response = df["User response produced by Prompt 2"].tolist()

In [ ]:
#You do not need to edit this cell. Just run it using the button on the left
dataset = []

for i in range(len(queries)):
    dataset.append(
        {
            "user_input":queries[i],
            "retrieved_contexts":retrieved_contexts[i],
            "response":response[i],
        }
    )

In [ ]:
#You do not need to edit this cell. Just run it using the button on the left
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset)


# Step 3. Evaluate

We have successfully collected the evaluation data. Now, we can evaluate our RAG system on the collected dataset using a set of commonly used RAG evaluation metrics.

We will be using metrics for faithfulness, response relevancy, and context utilization.

In [ ]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.0 MB/s eta 0:00:00


In [ ]:
#You do not need to edit this cell. Just run it using the button on the left
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from ragas.metrics import Faithfulness, ResponseRelevancy, ContextPrecision, ContextRecall, NoiseSensitivity, ContextEntityRecall
import os
import time

config = {
    "model": "gemini-2.0-flash-lite",
    "temperature": 0.4,
    "max_tokens": None,
    "top_p": 0.8,
}

# Initialize evaluator with Google AI Studio
evaluator_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model=config["model"],
    temperature=config["temperature"],
    max_tokens=config["max_tokens"],
    top_p=config["top_p"],
))

evaluator_embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",  # Google's text embedding model
    task_type="retrieval_document"  # Optional: specify the task type
))

results = []
for point in dataset:
    evaluation_dataset = EvaluationDataset.from_list([point])
    result = evaluate(dataset=evaluation_dataset, metrics=[Faithfulness(), ResponseRelevancy()], llm=evaluator_llm, embeddings=evaluator_embeddings)
    results.append(result)
    time.sleep(2)

print("-----------------------------------------------")
for i, result in enumerate(results):
    print("Query:", queries[i])
    print("\n")
    print("Faithfulness:", result['faithfulness'][0])
    print("Response Relevancy:", result['answer_relevancy'][0])
    #print("Context Utilization:", result['context_utilization'][0])
    print("-----------------------------------------------")

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

-----------------------------------------------
Query: Example Query: What skills and training did you have to go through in order to become a firefighter? Do you ever feel unprepared or scared when you respond to an emergency call?


Faithfulness: 0.6363636363636364
Response Relevancy: 0.8435428087428094
-----------------------------------------------
Query: What kind of personality do you need to work in public service or local government jobs?


Faithfulness: 0.9782608695652174
Response Relevancy: 0.9680376696812049
-----------------------------------------------
Query: I'm passionate about giving back and would love to get more involved in my community. What are some meaningful ways I can contribute?


Faithfulness: 1.0
Response Relevancy: 0.9010655089278176
-----------------------------------------------
Query: I'm feeling really stressed about my future and anxious that I might not succeed in my job. What steps can I take to manage these worries and build confidence?


Faithfulne

Now with these evaluations, copy over your results into the Google Sheet
